In [1]:
import rasterio
from rasterio.warp import reproject, Resampling
import numpy as np
import os

# 1. Reference grid (10 m master)
# This is the 'gold standard' grid all your data must match
reference = r"D:\landslide\final_data\distance_roads_tehri.tif"

with rasterio.open(reference) as ref:
    ref_meta = ref.meta.copy()
    ref_transform = ref.transform
    ref_crs = ref.crs
    ref_shape = (ref.height, ref.width)

# 2. Input: Your original 30m DEM
input_path = r"D:\landslide\tehri_landslide\data\DEM\DEM_Tehri_fixed.tif"
output_path = r"D:\landslide\final_data\Elevation_10m.tif"

# 3. Resample Function (Applied to Elevation)
with rasterio.open(input_path) as src:
    # Read elevation data
    data = src.read(1).astype(np.float32)
    
    # Standardize NoData to -9999 for Machine Learning
    nodata_val = -9999.0
    resampled = np.full(ref_shape, nodata_val, dtype=np.float32)

    reproject(
        source=data,
        destination=resampled,
        src_transform=src.transform,
        src_crs=src.crs,
        dst_transform=ref_transform,
        dst_crs=ref_crs,
        resampling=Resampling.bilinear, # Bilinear is best for smooth terrain heights
        src_nodata=src.nodata,
        dst_nodata=nodata_val
    )

# 4. Save the New 10m Elevation File
new_meta = ref_meta.copy()
new_meta.update(dtype=np.float32, nodata=nodata_val)

with rasterio.open(output_path, "w", **new_meta) as dst:
    dst.write(resampled, 1)

print(f"Elevation successfully resampled to 10m: {os.path.basename(output_path)}")

Elevation successfully resampled to 10m: Elevation_10m.tif


In [2]:
import rasterio
import os

folder = r"D:\landslide\final_data"
master_path = r"D:\landslide\final_data\distance_roads_tehri.tif"

with rasterio.open(master_path) as ref:
    ref_meta = ref.meta
    ref_bounds = ref.bounds

print(f"{'Raster Name':<30} | {'Shape Matches?':<15} | {'CRS Matches?'}")
print("-" * 65)

for file in os.listdir(folder):
    if file.endswith(".tif") and "_10m" in file:
        with rasterio.open(os.path.join(folder, file)) as src:
            shape_check = "PASS" if src.shape == ref.shape else "FAILED"
            crs_check = "PASS" if src.crs == ref.crs else "FAILED"
            print(f"{file:<30} | {shape_check:<15} | {crs_check}")

Raster Name                    | Shape Matches?  | CRS Matches?
-----------------------------------------------------------------
Aspect_deg_10m.tif             | PASS            | PASS
BSI_lite_Tehri_10m.tif         | PASS            | PASS
Curvature_10m.tif              | PASS            | PASS
Elevation_10m.tif              | PASS            | PASS
Geomorphon_clean_10m.tif       | PASS            | PASS
lithology_tehri_10m.tif        | PASS            | PASS
lulc_tehri_10m.tif             | PASS            | PASS
NDVI_Tehri_10m.tif             | PASS            | PASS
NDWI_Tehri_10m.tif             | PASS            | PASS
Rainfall_Tehri_clean_10m.tif   | PASS            | PASS
Slope_deg_10m.tif              | PASS            | PASS
Soil_Tehri_10m.tif             | PASS            | PASS
TRI_10m.tif                    | PASS            | PASS
TWI_clean_10m.tif              | PASS            | PASS


In [3]:
import rasterio
import numpy as np

path = r"D:\landslide\final_data\Elevation_10m.tif"

with rasterio.open(path) as src:
    data = src.read(1)
    nodata = src.nodata
    
    # Mask out NoData to get real statistics
    valid_data = data[data != nodata]
    
    print(f"--- Elevation Detail Report: {src.name} ---")
    print(f"Min Elevation: {np.min(valid_data):.2f} m")
    print(f"Max Elevation: {np.max(valid_data):.2f} m")
    print(f"Mean Elevation: {np.mean(valid_data):.2f} m")
    print(f"Standard Deviation: {np.std(valid_data):.2f} m")
    print(f"Total Valid Pixels: {len(valid_data)}")
    print(f"Total NoData Pixels: {np.sum(data == nodata)}")
    
    # Check for anomalies
    if np.min(valid_data) < 0:
        print("ALERT: Negative elevation found. Check for sea-level errors.")
    if np.max(valid_data) > 8848:
        print("ALERT: Elevation higher than Everest. Check for sensor noise.")

--- Elevation Detail Report: D:\landslide\final_data\Elevation_10m.tif ---
Min Elevation: 329.00 m
Max Elevation: 6711.00 m
Mean Elevation: 1912.69 m
Standard Deviation: 973.79 m
Total Valid Pixels: 43913554
Total NoData Pixels: 53657690
